In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI 
from typing import TypedDict, Literal 
from dotenv import load_dotenv 
from pydantic import BaseModel, Field

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI(model='gpt-4o-mini')

In [4]:
class sentimentSchema(BaseModel):

    sentiment: Literal["positive", "negative"] = Field(description='sentiment of the review')

In [ ]:
class DiagnosisSchema(BaseModel):
    

In [5]:
structured_model = model.with_structured_output(sentimentSchema)

In [6]:
prompt = 'what is the sentiment of the following review'
structured_model.invoke(prompt).sentiment

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-or-v1*************************************************************38a4. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [8]:
class ReviewState(TypedDict):

    review: str
    sentiment: Literal["positive", "negative"]
    diagonsis: dict
    reposne: str


In [ ]:
graph = StateGraph(ReviewState)

def find_sentiment(state: ReviewState):
     prompt = f'For the following review find out the sentiment \n {state["review"]}'
     sentiment = structured_model. invoke(prompt). sentiment

     return {'sentiment': sentiment}


def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:

    if state['sentiment']=='positive':
        return 'positive response'
    else:
        return 'run_diagnosis'

def positive_response(state: ReviewState):

    prompt = f"""Write a warm thank-you message in response to this review: \n\n\"{state['review']}\"\n Also, kindly ask the user to leave feedback on our website. """

    response = model.invoke(prompt).content

    return {'response': response}
    

In [ ]:
graph.add_node('final_sentiment', find_sentiment)
graph. add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')
graph. add_edge('find_sentiment', END)